# Sentinel Edge: the Strands Decider 2B decision pattern, step by step

This notebook shows the one decision Sentinel Edge hands to **Strands Decider 2B**: *how should the control room treat this new alarm?* (escalate, review or suppress).

It runs entirely on the edge device. No AWS account is needed, only a Decider server on this machine:

```bash
pip install strands-decider pyyaml jupyter          # add [cuda] for a GPU; it also runs on CPU, just slower
strands-decider serve StrandsAgents/strands-decider-2B-qwen3.5-v1-2610 --port 8099
```

1. **The reusable gate** (`decider_policy/`) on something that isn't a factory at all.
2. **One alarm, traced** through scoped context, the four yes/no signals, the category and the route.
3. **A whole alarm flood** from the bottling-line profile, compared with the labels.
4. **Failure goes to a human:** what happens when Decider is unreachable.

In [1]:
import os, sys
from pathlib import Path

# Use the local Decider server. Set these before importing decider_policy: it fills
# unset variables from the repo's .env, which may point at the hosted API.
os.environ["STRANDS_DECIDER_URL"] = "http://127.0.0.1:8099"
os.environ["STRANDS_DECIDER_API_KEY"] = ""
os.environ["STRANDS_DECIDER_RPM"] = "0"

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from decider_policy import DeciderClient, PolicyGate
from alarm_triage import PlantTriage, load_all, make_alarm

client = DeciderClient()
print("Decider reachable:", client.health())

Decider reachable: True


## 1. The reusable gate, outside manufacturing

`PolicyGate` knows nothing about factories. You give it:

- **signals:** yes/no questions, each about *one* property. Decider answers each with a calibrated probability.
- **classify(signals):** your code turns the probabilities into a category.
- **route(category, signals):** your code turns the category into an action.

Decider measures; Python decides. Here the same gate triages support tickets.

In [2]:
TICKET_SIGNALS = {
    "outage":  "Does the ticket report that a production service is down or unusable for customers right now?",
    "billing": "Is the ticket mainly about a charge, invoice or refund?",
    "spam":    "Is the ticket spam, an advert, or unrelated to the product?",
}

def classify_ticket(s):
    if s["outage"] >= 0.5:
        return "incident", s["outage"]
    if s["spam"] >= 0.6:
        return "spam", s["spam"]
    if s["billing"] >= 0.5:
        return "billing", s["billing"]
    return "general", 1 - max(s.values())

def route_ticket(category, s):
    return {"incident": "page_on_call", "spam": "close", "billing": "billing_queue"}.get(category, "support_queue")

ticket_gate = PolicyGate(TICKET_SIGNALS, classify=classify_ticket, route=route_ticket, client=client, fallback_route="support_queue")

tickets = [
    "Checkout returns HTTP 500 for every customer since 10:05, we are losing orders!",
    "I was charged twice for my March invoice, please refund one of them.",
    "Cheap SEO backlinks!!! Rank #1 on Google in 7 days, reply now.",
    "How do I change the email address on my account?",
]
for t in tickets:
    d = ticket_gate.decide(t)
    sig = "  ".join(f"{k}={v:.2f}" for k, v in d.signals.items())
    print(f"{d.route:<14} {d.category:<9} [{sig}]  {d.latency_ms:5.0f} ms  | {t[:60]}")

page_on_call   incident  [outage=0.65  billing=0.24  spam=0.24]    334 ms  | Checkout returns HTTP 500 for every customer since 10:05, we


billing_queue  billing   [outage=0.07  billing=0.83  spam=0.25]    265 ms  | I was charged twice for my March invoice, please refund one 


close          spam      [outage=0.12  billing=0.31  spam=0.68]    287 ms  | Cheap SEO backlinks!!! Rank #1 on Google in 7 days, reply no


support_queue  general   [outage=0.09  billing=0.09  spam=0.30]    213 ms  | How do I change the email address on my account?


## 2. One alarm, step by step

The bottling line's pasteuriser loses temperature. Under-pasteurised juice is a food-safety hazard, so this alarm must reach the operator.

Sentinel Edge asks four yes/no signals. Each signal only sees the context it needs (this **scoping** took held-out accuracy from 57-60% to 79-80%):

| signal | sees |
|---|---|
| danger, noise | the new alarm only |
| maintenance | the alarm plus the operator log |
| consequence | the alarm plus every alarm already active |

In [3]:
profiles = load_all()
bottling = profiles["bottling_line"]
scenario = bottling.scenario("pasteuriser_underheat")
print(scenario["title"])

triage = PlantTriage(bottling, client=client)
for note in scenario.get("oplog", []):
    triage.state.add_oplog(note)

# play the scenario up to the pasteuriser alarm, so the cascade context is realistic
events = scenario["events"]
target = next(i for i, ev in enumerate(events) if ev["tag"] == "TT-PAST-LO")
for i, ev in enumerate(events[:target]):
    triage.decide(make_alarm(bottling, ev["tag"], "nb", i, expected=ev.get("expected")))

alarm = make_alarm(bottling, "TT-PAST-LO", "nb", target, expected=events[target].get("expected"))
d, states = triage.decide(alarm)

for scope, text in states.items():
    print(f"\n--- context for scope '{scope}' ({len(text)} chars) ---")
    print(text[:600] + (" ..." if len(text) > 600 else ""))

Pasteuriser loses temperature -> diversion starves the whole line



--- context for scope 'alarm' (449 chars) ---
Plant: PET Bottling Line 2 (Food & beverage packaging). High-speed juice bottling: a pasteuriser heats product, a rotary filler fills bottles, a capper seals them, and a checkweigher and labeller finish the line. Product temperature in the pasteuriser is a food-safety critical control point.

NEW ALARM: TT-PAST-LO [critical] "Pasteuriser hold-tube temperature below 72 C (food-safety CCP)"
  equipment PAST-01 (flash pasteuriser) in Pasteurisation

--- context for scope 'oplog' (597 chars) ---
Plant: PET Bottling Line 2 (Food & beverage packaging). High-speed juice bottling: a pasteuriser heats product, a rotary filler fills bottles, a capper seals them, and a checkweigher and labeller finish the line. Product temperature in the pasteuriser is a food-safety critical control point.

NEW ALARM: TT-PAST-LO [critical] "Pasteuriser hold-tube temperature below 72 C (food-safety CCP)"
  equipment PAST-01 (flash pasteuriser) in Pasteurisation

OPERA

In [4]:
print("Decider signals (P = yes):")
for name, p in d.signals.items():
    print(f"  {name:<12} {p:.2f}  " + "#" * round(p * 30))
print(f"\ncategory : {d.category}  (confidence {d.confidence:.2f})")
print(f"route    : {d.route}")
print(f"expected : {alarm.get('expected')}")
print(f"latency  : {d.latency_ms:.0f} ms on {d.model}")

Decider signals (P = yes):
  danger       0.75  ######################
  noise        0.17  #####
  consequence  0.36  ###########
  maintenance  0.18  #####

category : safety_critical  (confidence 0.75)
route    : escalate
expected : safety_critical
latency  : 1167 ms on strands-decider-2B-qwen3.5-v1-2610


## 3. A whole alarm flood

Now the full scenario, from a clean state, the same way the edge agent runs it. `escalate` sends the alarm to the Strands agent on Amazon Bedrock for a root cause; `review` shows it to the operator without an LLM call; `suppress` keeps it off the screen.

In [5]:
triage = PlantTriage(bottling, client=client)
for note in scenario.get("oplog", []):
    triage.state.add_oplog(note)

rows, hits = [], 0
for i, ev in enumerate(scenario["events"]):
    a = make_alarm(bottling, ev["tag"], "nb", i, expected=ev.get("expected"))
    d, _ = triage.decide(a)
    ok = d.category == ev.get("expected")
    hits += ok
    sig = " ".join(f"{k[0]}={v:.2f}" for k, v in d.signals.items())
    rows.append(f"{ev['tag']:<12} {sig}  {d.category:<20} -> {d.route:<8} {d.latency_ms:5.0f} ms  expected={ev.get('expected'):<20} {'OK' if ok else 'MISS'}")

print("signals: d=danger c=consequence n=noise m=maintenance\n")
print("\n".join(rows))
print(f"\n{hits}/{len(rows)} categories match the labels")

signals: d=danger c=consequence n=noise m=maintenance

LAB-GAP      d=0.16 n=0.93 c=0.25 m=0.24  nuisance             -> suppress   831 ms  expected=nuisance             OK
CIP-COND     d=0.24 n=0.23 c=0.33 m=0.24  actionable           -> escalate   855 ms  expected=nuisance             MISS
TT-PAST-LO   d=0.75 n=0.17 c=0.36 m=0.18  safety_critical      -> escalate   915 ms  expected=safety_critical      OK
FDV-DIVERT   d=0.33 n=0.19 c=0.51 m=0.20  duplicate_of_active  -> review     864 ms  expected=duplicate_of_active  OK
LT-BUF-LO    d=0.22 n=0.18 c=0.44 m=0.19  actionable           -> escalate  1053 ms  expected=duplicate_of_active  MISS
LAB-GAP      d=0.16 n=0.93 c=0.54 m=0.24  nuisance             -> suppress  1117 ms  expected=nuisance             OK
FIL-STARVED  d=0.25 n=0.21 c=0.53 m=0.20  duplicate_of_active  -> review     894 ms  expected=duplicate_of_active  OK
PAL-IDLE     d=0.11 n=0.30 c=0.49 m=0.22  duplicate_of_active  -> review     906 ms  expected=duplicate_of_active  

A miss is not the same as a hidden alarm. The router only suppresses when Decider is *confident* an alarm is noise or a duplicate, and an alarm whose danger signal sits in the uncertain band is never suppressed. Mistakes therefore tend to land in `escalate` or `review`, where a person still sees them.

## 4. Failure goes to a human

If Decider is down, busy or returns something malformed, the gate does not guess and does not drop the alarm: it returns the fallback route, `review`, and marks the decision as degraded.

In [6]:
dead = DeciderClient(url="http://127.0.0.1:9")   # nothing listens on port 9
offline = PlantTriage(bottling, client=dead)
d, _ = offline.decide(make_alarm(bottling, "TT-PAST-LO", "nb", 0))
print(f"route={d.route}  category={d.category}  degraded={d.degraded}")

route=review  category=unclassified  degraded=True


## Using it in a Strands agent

`PolicyGateIntervention` wraps the same gate as a Strands `InterventionHandler`, so blocked routes never reach the model:

```python
from strands import Agent
from decider_policy import PolicyGateIntervention

agent = Agent(model=bedrock_model,
              interventions=[PolicyGateIntervention(gate, deny_routes={"suppress"})])
```

Repo: https://github.com/Vishalghost/Sentinel-Edge · Strands Decider: https://github.com/strands-labs/strands-decider